In [1]:
# Step 6 - First cell: load the data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

red = pd.read_csv("../data/raw/winequality-red.csv", sep=";")
white = pd.read_csv("../data/raw/winequality-white.csv", sep=";")

print("red:  ", red.shape)
print("white:", white.shape)

red.head()

red:   (1599, 12)
white: (4898, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [2]:
# Step 7 - Combine, then check for missing values and duplicates

wine = pd.concat(
    [red.assign(wine_type="red"), white.assign(wine_type="white")],
    ignore_index=True
)

print("combined:", wine.shape)
print("missing values:", wine.isnull().sum().sum())
print("duplicate rows:", wine.duplicated().sum())

combined: (6497, 13)
missing values: 0
duplicate rows: 1177


In [3]:
# Step 8 — Target distribution: what does "quality" actually look like?

counts = wine["quality"].value_counts().sort_index()
pct = wine["quality"].value_counts(normalize=True).sort_index() * 100

print(pd.DataFrame({"count": counts, "percent": pct.round(1)}))
print()
print("range:", wine["quality"].min(), "to", wine["quality"].max())
print("mean: ", round(wine["quality"].mean(), 2))

         count  percent
quality                
3           30      0.5
4          216      3.3
5         2138     32.9
6         2836     43.7
7         1079     16.6
8          193      3.0
9            5      0.1

range: 3 to 9
mean:  5.82


In [4]:
# Step 9 — Correlation of each feature with quality

corr = wine.corr(numeric_only=True)["quality"].drop("quality")
print(corr.sort_values(ascending=False).round(3))

alcohol                 0.444
citric acid             0.086
free sulfur dioxide     0.055
sulphates               0.038
pH                      0.020
residual sugar         -0.037
total sulfur dioxide   -0.041
fixed acidity          -0.077
chlorides              -0.201
volatile acidity       -0.266
density                -0.306
Name: quality, dtype: float64


In [5]:
# Step 10 — Correlations split by wine type, plus the alcohol/density check

corr_red = red.corr(numeric_only=True)["quality"].drop("quality")
corr_white = white.corr(numeric_only=True)["quality"].drop("quality")

comparison = pd.DataFrame({
    "red": corr_red,
    "white": corr_white,
    "combined": corr
}).round(3).sort_values("combined", ascending=False)

print(comparison)
print()
print("alcohol vs density:", round(wine["alcohol"].corr(wine["density"]), 3))
print("mean quality — red:", round(red["quality"].mean(), 2),
      "| white:", round(white["quality"].mean(), 2))

                        red  white  combined
alcohol               0.476  0.436     0.444
citric acid           0.226 -0.009     0.086
free sulfur dioxide  -0.051  0.008     0.055
sulphates             0.251  0.054     0.038
pH                   -0.058  0.099     0.020
residual sugar        0.014 -0.098    -0.037
total sulfur dioxide -0.185 -0.175    -0.041
fixed acidity         0.124 -0.114    -0.077
chlorides            -0.129 -0.210    -0.201
volatile acidity     -0.391 -0.195    -0.266
density              -0.175 -0.307    -0.306

alcohol vs density: -0.687
mean quality — red: 5.64 | white: 5.88


# # Phase 1.B — Data Acquisition & Exploration

## Dataset

- **Source:** UCI Machine Learning Repository, Wine Quality (Cortez et al., Portugal — Vinho Verde)
- **Files:** `winequality-red.csv` (1,599 rows), `winequality-white.csv` (4,898 rows)
- **Combined:** 6,497 rows × 12 columns (11 chemical features + `quality`), plus an added `wine_type` column
- **Delimiter:** semicolon, not comma — despite the `.csv` extension (European convention)
- **Missing values:** none
- **Duplicate rows:** 1,177 (18%) — plausibly distinct bottles with identical measurements, not corruption

## Target variable — `quality`

Integer score, median of at least three sensory assessors.

| quality | count | percent |
|---:|---:|---:|
| 3 | 30 | 0.5% |
| 4 | 216 | 3.3% |
| 5 | 2,138 | 32.9% |
| 6 | 2,836 | 43.7% |
| 7 | 1,079 | 16.6% |
| 8 | 193 | 3.0% |
| 9 | 5 | 0.1% |

Range 3–9 (never 0–2 or 10). Mean 5.82. **Quality 5 and 6 alone account for 77% of all wine.**
The extremes are starved: 5 examples at quality 9, 30 at quality 3.

## Decision — regression, not classification

Predict the quality score as a continuous number.

**Rationale:**

- The target is **ordinal** — 3 < 4 < 5 < ... — and regression preserves that. Multiclass
  classification treats the seven scores as unordered names, scoring "predicted 8, actual 9"
  as exactly as wrong as "predicted 3, actual 9".
- **Multiclass is not viable regardless:** quality 9 has 5 examples, which is one wine per fold
  under 5-fold cross-validation. The model would silently collapse into predicting only 5, 6 and 7.
- Binary (good ≥ 7 vs rest) was the alternative. Rejected because it requires an arbitrary
  cutoff and discards the distinction between a 3 and a 6.
- Regression does **not** limit us to linear models. `RandomForestRegressor` captures
  non-linear relationships and feature interactions exactly as well as a classifier does —
  only the leaf contents differ (an average rather than a class vote).

## Correlation analysis

Correlation with `quality`, computed separately by wine type to avoid the pooled view masking
type-specific effects (white is 75% of the combined dataset and dominates it).

| feature | red | white | combined |
|---|---:|---:|---:|
| alcohol | 0.476 | 0.436 | 0.444 |
| sulphates | 0.251 | 0.054 | 0.038 |
| citric acid | 0.226 | −0.009 | 0.086 |
| fixed acidity | 0.124 | −0.114 | −0.077 |
| residual sugar | 0.014 | −0.098 | −0.037 |
| free sulfur dioxide | −0.051 | 0.008 | 0.055 |
| pH | −0.058 | 0.099 | 0.020 |
| chlorides | −0.129 | −0.210 | −0.201 |
| density | −0.175 | −0.307 | −0.306 |
| total sulfur dioxide | −0.185 | −0.175 | −0.041 |
| volatile acidity | −0.391 | −0.195 | −0.266 |

**1. Alcohol is the strongest single predictor and it is robust** — 0.476 in red, 0.436 in white.
Squared, r² ≈ 0.20: a straight-line fit on alcohol alone accounts for about 20% of the variation
in quality. Substantial for a human sensory rating. Note this is statistical association, not
causation — riper grapes yield both more sugar (hence alcohol) and more concentrated flavour.

**2. The pooled view is misleading.** Density reads −0.306 combined, which is essentially white's
−0.307; red's much weaker −0.175 is outvoted by white's 3:1 row advantage. Always check pooled
correlations against subgroups.

**3. Multicollinearity between alcohol and density**: r = −0.687, i.e. 47% shared variation.
Alcohol is less dense than water, so this is close to mechanical. It destabilises linear model
coefficients (the two features compete for credit) but does not meaningfully harm a Random Forest.

**4. Several features reverse sign between red and white.** Fixed acidity helps reds (+0.124) and
hurts whites (−0.114). Sulphates matter roughly five times more in red. Volatile acidity is twice
as damaging in red. This is a textbook **interaction**: a feature's effect depends on the value of
another feature — here, wine type.

## Implications for 1.C and 1.D

- **One model on all 6,497 rows, with `wine_type` as a 12th input feature** — not two separate
  models. A tree can split on wine type and learn different rules per branch, which is exactly the
  behaviour observed above. Separate models would waste red's small sample and duplicate effort.
- **Testable prediction for 1.D:** linear regression will underperform Random Forest here, because
  a linear model cannot represent a feature that helps one wine type and hurts the other without
  explicit interaction terms.
- **Deployment consequence:** the prediction API takes **12 inputs, not 11**. `wine_type` must be
  part of the request payload and encoded consistently at training and inference time.
- **Open item for 1.C:** decide how to handle the 1,177 duplicate rows. The risk is leakage — the
  same row landing in both training and holdout sets, letting the model be graded on wine it has
  already memorised.
- **Open item for 1.C:** outlier treatment. Chemical measurements have long right tails
  (residual sugar, chlorides, sulfur dioxide) that warrant inspection.